In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt

from scipy.special import ellipk

from ipywidgets import (
    FloatSlider,
    RadioButtons,
    HTML,
    HTMLMath,
    VBox,
    HBox,
    Layout
)

from IPython.display import display

# ============================================================
# FUNDAMENTAL RECTANGLE, ZEROS AND POLES
# OF JACOBI ELLIPTIC FUNCTIONS
#
# Functions:
#
#   sn(u,k)
#   cd(u,k) = cn(u,k) / dn(u,k)
#
# Normalized coordinates:
#
#   X = Re(u) / K
#   Y = Im(u) / K'
#
# scipy.special uses m = k^2
# ============================================================

plt.ioff()

# ============================================================
# JUPYTER / BINDER DISPLAY SETTINGS
# ============================================================

display(HTML("""
<style>

.container {
    width: 98% !important;
    max-width: none !important;
}

.output_area,
.output_subarea {
    max-width: none !important;
    height: auto !important;
    max-height: none !important;
    overflow: visible !important;
}

.output_scroll {
    height: auto !important;
    max-height: none !important;
    overflow: visible !important;
    box-shadow: none !important;
}

.jp-Cell-outputWrapper,
.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output {
    max-width: none !important;
    height: auto !important;
    max-height: none !important;
    overflow: visible !important;
}

.widget-output,
.jupyter-widgets-output-area,
.widget-box {
    max-width: none !important;
    height: auto !important;
    max-height: none !important;
    overflow: visible !important;
}

.jupyter-matplotlib,
.jupyter-matplotlib-figure {
    overflow: visible !important;
    resize: none !important;
}

.jupyter-matplotlib::-webkit-resizer,
.jupyter-matplotlib-figure::-webkit-resizer {
    display: none !important;
}

.fr-title {
    font-family: Arial, sans-serif;
    font-size: 20px;
    font-weight: bold;
    color: #6f3fa0;
}

.fr-label {
    font-family: Arial, sans-serif;
    font-size: 14px;
    font-weight: bold;
}

.fr-value {
    font-family: Arial, sans-serif;
    font-size: 14px;
    font-weight: bold;
    color: #0b3d91;
}

.fr-radio .widget-radio-box {
    display: flex !important;
    flex-direction: row !important;
    flex-wrap: nowrap !important;
    gap: 22px !important;
    align-items: center !important;
}

.fr-radio .widget-radio-box label {
    margin: 0 !important;
    white-space: nowrap !important;
}

.fr-radio > label {
    display: none !important;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

documentation = HTML("""
<div style="
    width:1180px;
    font-family:Arial, sans-serif;
    font-size:15px;
    line-height:1.48;
    margin-bottom:10px;
">

<div class="fr-title" style="margin-bottom:8px;">
Fundamental Rectangle, Zeros and Poles
</div>

<div style="margin-bottom:5px;">
The zeros & poles of Jacobi elliptic functions form regular
two-dimensional lattices on the complex u-plane. Their spacing
is determined by the complete elliptic integrals K and K′.
</div>

<div style="margin-bottom:5px;">
The axes are normalized as Re(u)/K and Im(u)/K′. Therefore,
the geometrical structure of the lattice remains fixed while
the actual dimensions K and K′ vary with modulus k.
</div>

<div>
<b>This notebook:</b> displays the zeros and poles of sn(u,k)
and cd(u,k), together with the fundamental rectangle defined
by the points s, c, n and d.
</div>

</div>
""")

# ============================================================
# FUNCTION SELECTOR
# ============================================================

function_selector = RadioButtons(
    options=[
        ('sn(u,k)', 'sn'),
        ('cd(u,k)', 'cd')
    ],
    value='sn',
    description='',
    layout=Layout(
        width='300px'
    )
)

function_selector.add_class(
    'fr-radio'
)

# ============================================================
# k SLIDER
# ============================================================

slider_style = {
    'description_width': '0px'
}

k_slider = FloatSlider(
    min=0.05,
    max=0.95,
    step=0.05,
    value=0.70,
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=Layout(
        width='235px'
    )
)

k_label = HTML(
    '<div class="fr-label">Elliptic modulus k:</div>',
    layout=Layout(
        width='145px',
        min_width='145px'
    )
)

k_value = HTML(
    '<div class="fr-value">0.70</div>',
    layout=Layout(
        width='65px',
        min_width='65px',
        margin='0px 0px 0px 6px'
    )
)

k_row = HBox(
    [
        k_label,
        k_slider,
        k_value
    ],
    layout=Layout(
        width='470px',
        height='40px',
        align_items='center'
    )
)

# ============================================================
# CONTROLS PANEL
# ============================================================

controls_panel = VBox(
    [
        HTML("""
        <div class="fr-title" style="margin-bottom:8px;">
            Parameters
        </div>
        """),

        HTML("""
        <div class="fr-label" style="margin-bottom:5px;">
            Elliptic function
        </div>
        """),

        function_selector,

        HTML("""
        <div style="height:5px;"></div>
        """),

        k_row
    ],
    layout=Layout(
        width='500px',
        padding='10px 14px',
        border='1px solid #d2c2df',
        overflow='visible'
    )
)

# ============================================================
# CURRENT VALUES
# ============================================================

kp_math = HTMLMath()
K_math = HTMLMath()
Kp_math = HTMLMath()

current_values_panel = VBox(
    [
        HTML("""
        <div class="fr-title" style="margin-bottom:8px;">
            Elliptic Parameters
        </div>
        """),

        HBox(
            [
                kp_math,
                K_math,
                Kp_math
            ],
            layout=Layout(
                width='610px',
                gap='20px',
                overflow='visible'
            )
        )
    ],
    layout=Layout(
        width='640px',
        padding='10px 14px',
        border='1px solid #d2c2df',
        overflow='visible'
    )
)

# ============================================================
# TOP ROW
# ============================================================

top_row = HBox(
    [
        controls_panel,
        current_values_panel
    ],
    layout=Layout(
        width='1160px',
        gap='15px',
        align_items='stretch',
        overflow='visible'
    )
)

# ============================================================
# COMPLEX PLANE FIGURE
# ============================================================

fig, ax = plt.subplots(
    figsize=(7.4, 5.4)
)

fig.canvas.header_visible = False
fig.canvas.footer_visible = False
fig.canvas.toolbar_visible = False

fig.canvas.layout = Layout(
    width='740px',
    height='540px',
    overflow='visible'
)

ax.set_title(
    'Zeros and Poles on the Complex u-Plane',
    fontsize=14,
    fontweight='bold',
    color='#6f3fa0'
)

ax.set_xlabel(
    'Re(u) / K',
    fontsize=11
)

ax.set_ylabel(
    'Im(u) / K′',
    fontsize=11
)

ax.set_xlim(
    -4.5,
    4.5
)

ax.set_ylim(
    -3.5,
    3.5
)

ax.set_aspect(
    'equal',
    adjustable='box'
)

ax.axhline(
    0.0,
    color='black',
    linewidth=1.0
)

ax.axvline(
    0.0,
    color='black',
    linewidth=1.0
)

ax.grid(
    True,
    linestyle=':',
    alpha=0.35
)

# ============================================================
# GRID LINES
#
# Integer multiples of K and K'
# ============================================================

for x_grid in range(
    -4,
    5
):

    ax.axvline(
        x_grid,
        linewidth=0.7,
        linestyle=':',
        alpha=0.25
    )

for y_grid in range(
    -3,
    4
):

    ax.axhline(
        y_grid,
        linewidth=0.7,
        linestyle=':',
        alpha=0.25
    )

# ============================================================
# FUNDAMENTAL RECTANGLE
#
# s = (0,0)
# c = (K,0)
# n = (0,iK')
# d = (K,iK')
#
# In normalized coordinates:
#
# s = (0,0)
# c = (1,0)
# n = (0,1)
# d = (1,1)
# ============================================================

rectangle_x = [
    0.0,
    1.0,
    1.0,
    0.0,
    0.0
]

rectangle_y = [
    0.0,
    0.0,
    1.0,
    1.0,
    0.0
]

fundamental_rectangle, = ax.plot(
    rectangle_x,
    rectangle_y,
    linewidth=3.0,
    label='Fundamental rectangle'
)

# ============================================================
# FUNDAMENTAL-RECTANGLE VERTICES
# ============================================================

vertex_points, = ax.plot(
    [
        0.0,
        1.0,
        0.0,
        1.0
    ],
    [
        0.0,
        0.0,
        1.0,
        1.0
    ],
    linestyle='None',
    marker='o',
    markersize=6
)

ax.text(
    0.07,
    0.07,
    's',
    fontsize=11,
    fontweight='bold'
)

ax.text(
    1.07,
    0.07,
    'c',
    fontsize=11,
    fontweight='bold'
)

ax.text(
    0.07,
    1.07,
    'n',
    fontsize=11,
    fontweight='bold'
)

ax.text(
    1.07,
    1.07,
    'd',
    fontsize=11,
    fontweight='bold'
)

# ============================================================
# ZERO AND POLE MARKERS
#
# Created once. Their positions are updated according to
# the selected function.
# ============================================================

zero_points, = ax.plot(
    [],
    [],
    linestyle='None',
    marker='o',
    markersize=8,
    fillstyle='none',
    markeredgewidth=2.0,
    label='Zeros'
)

pole_points, = ax.plot(
    [],
    [],
    linestyle='None',
    marker='x',
    markersize=9,
    markeredgewidth=2.0,
    label='Poles'
)

# ============================================================
# LEGEND BELOW FIGURE
# ============================================================

ax.legend(
    loc='upper center',
    bbox_to_anchor=(0.5, -0.12),
    ncol=3,
    fontsize=9,
    frameon=True
)

fig.subplots_adjust(
    left=0.10,
    right=0.97,
    top=0.90,
    bottom=0.20
)

# ============================================================
# EQUATIONS PANEL
# ============================================================

zero_equation_math = HTMLMath()
pole_equation_math = HTMLMath()

equations_panel = VBox(
    [
        HTML("""
        <div class="fr-title" style="margin-bottom:8px;">
            Zero and Pole Lattices
        </div>
        """),

        HTML("""
        <div style="
            font-family:Arial;
            font-size:13px;
            font-weight:bold;
            color:#6f3fa0;
            margin-bottom:3px;
        ">
            Zeros
        </div>
        """),

        zero_equation_math,

        HTML("""
        <div style="
            font-family:Arial;
            font-size:13px;
            font-weight:bold;
            color:#6f3fa0;
            margin-top:10px;
            margin-bottom:3px;
        ">
            Poles
        </div>
        """),

        pole_equation_math
    ],
    layout=Layout(
        width='390px',
        padding='10px 14px',
        border='1px solid #d2c2df',
        overflow='visible'
    )
)

# ============================================================
# INTERPRETATION PANEL
# ============================================================

interpretation_html = HTML()

interpretation_panel = VBox(
    [
        interpretation_html
    ],
    layout=Layout(
        width='390px',
        padding='10px 14px',
        border='1px solid #d7c7e5',
        margin='10px 0px 0px 0px',
        overflow='visible'
    )
)

# ============================================================
# RIGHT COLUMN
# ============================================================

right_column = VBox(
    [
        equations_panel,
        interpretation_panel
    ],
    layout=Layout(
        width='400px',
        overflow='visible'
    )
)

# ============================================================
# VISUAL ROW
# ============================================================

visual_row = HBox(
    [
        fig.canvas,
        right_column
    ],
    layout=Layout(
        width='1160px',
        gap='15px',
        align_items='flex-start',
        overflow='visible'
    )
)

# ============================================================
# LATTICE GENERATION
#
# All coordinates returned below are NORMALIZED:
#
# X = Re(u)/K
# Y = Im(u)/K'
# ============================================================

def lattice_points(function_name):

    zero_x = []
    zero_y = []

    pole_x = []
    pole_y = []

    # --------------------------------------------------------
    # Integer lattice indices
    # --------------------------------------------------------

    m_values = range(
        -3,
        4
    )

    n_values = range(
        -2,
        3
    )

    # ========================================================
    # sn(u,k)
    #
    # zeros:
    #
    # z_mn = 2mK + 2niK'
    #
    # poles:
    #
    # p_mn = 2mK + (2n+1)iK'
    #
    # Normalized:
    #
    # zeros -> (2m, 2n)
    # poles -> (2m, 2n+1)
    # ========================================================

    if function_name == 'sn':

        for m in m_values:

            for n in n_values:

                zx = (
                    2 * m
                )

                zy = (
                    2 * n
                )

                px = (
                    2 * m
                )

                py = (
                    2 * n + 1
                )

                if (
                    -4.5 <= zx <= 4.5
                    and
                    -3.5 <= zy <= 3.5
                ):

                    zero_x.append(
                        zx
                    )

                    zero_y.append(
                        zy
                    )

                if (
                    -4.5 <= px <= 4.5
                    and
                    -3.5 <= py <= 3.5
                ):

                    pole_x.append(
                        px
                    )

                    pole_y.append(
                        py
                    )

    # ========================================================
    # cd(u,k)
    #
    # From the theory:
    #
    # zeros:
    #
    # z_mn = (2m+1)K + 2niK'
    #
    # poles:
    #
    # p_mn = (2m+1)K + (2n+1)iK'
    #
    # Normalized:
    #
    # zeros -> (2m+1, 2n)
    # poles -> (2m+1, 2n+1)
    # ========================================================

    else:

        for m in m_values:

            for n in n_values:

                zx = (
                    2 * m + 1
                )

                zy = (
                    2 * n
                )

                px = (
                    2 * m + 1
                )

                py = (
                    2 * n + 1
                )

                if (
                    -4.5 <= zx <= 4.5
                    and
                    -3.5 <= zy <= 3.5
                ):

                    zero_x.append(
                        zx
                    )

                    zero_y.append(
                        zy
                    )

                if (
                    -4.5 <= px <= 4.5
                    and
                    -3.5 <= py <= 3.5
                ):

                    pole_x.append(
                        px
                    )

                    pole_y.append(
                        py
                    )

    return (
        np.array(zero_x),
        np.array(zero_y),
        np.array(pole_x),
        np.array(pole_y)
    )

# ============================================================
# UPDATE FUNCTION
#
# No clear_output()
# No figure recreation
# No axis changes
#
# Only marker coordinates and displayed text are updated.
# ============================================================

def update_notebook(change=None):

    mode = (
        function_selector.value
    )

    k = (
        k_slider.value
    )

    # ========================================================
    # ELLIPTIC PARAMETERS
    # ========================================================

    m_parameter = (
        k**2
    )

    kp = np.sqrt(
        1.0
        -
        k**2
    )

    K = ellipk(
        m_parameter
    )

    Kp = ellipk(
        1.0 - m_parameter
    )

    # ========================================================
    # SLIDER VALUE
    # ========================================================

    k_value.value = (
        f'<div class="fr-value">{k:.2f}</div>'
    )

    # ========================================================
    # PARAMETER VALUES
    #
    # Separate widgets are used.
    # No \quad or \qquad commands.
    # ========================================================

    kp_math.value = (
        r'\('
        r'k^{\prime}='
        +
        f'{kp:.6f}'
        +
        r'\)'
    )

    K_math.value = (
        r'\('
        r'K='
        +
        f'{K:.6f}'
        +
        r'\)'
    )

    Kp_math.value = (
        r'\('
        r'K^{\prime}='
        +
        f'{Kp:.6f}'
        +
        r'\)'
    )

    # ========================================================
    # LATTICE POINTS
    # ========================================================

    (
        zero_x,
        zero_y,
        pole_x,
        pole_y
    ) = lattice_points(
        mode
    )

    zero_points.set_data(
        zero_x,
        zero_y
    )

    pole_points.set_data(
        pole_x,
        pole_y
    )

    # ========================================================
    # EQUATIONS AND INTERPRETATION
    # ========================================================

    if mode == 'sn':

        zero_equation_math.value = (
            r'\('
            r'z_{mn}'
            r'='
            r'2mK'
            r'+'
            r'2niK^{\prime}'
            r'\)'
        )

        pole_equation_math.value = (
            r'\('
            r'p_{mn}'
            r'='
            r'2mK'
            r'+'
            r'(2n+1)iK^{\prime}'
            r'\)'
        )

        interpretation_html.value = """
        <div style="
            font-family:Arial, sans-serif;
            font-size:14px;
            line-height:1.52;
        ">

        <div style="
            color:#6f3fa0;
            font-size:17px;
            font-weight:bold;
            margin-bottom:6px;
        ">
        Interpretation
        </div>

        <div style="margin-bottom:6px;">
        For <b>sn(u,k)</b>, zeros occur at even multiples
        of K and iK′.
        </div>

        <div style="margin-bottom:6px;">
        The poles lie halfway between successive rows of zeros
        in the imaginary direction.
        </div>

        <div>
        The complete lattice is repeated according to the real
        and imaginary periods of the Jacobi elliptic function.
        </div>

        </div>
        """

    else:

        zero_equation_math.value = (
            r'\('
            r'z_{mn}'
            r'='
            r'(2m+1)K'
            r'+'
            r'2niK^{\prime}'
            r'\)'
        )

        pole_equation_math.value = (
            r'\('
            r'p_{mn}'
            r'='
            r'(2m+1)K'
            r'+'
            r'(2n+1)iK^{\prime}'
            r'\)'
        )

        interpretation_html.value = """
        <div style="
            font-family:Arial, sans-serif;
            font-size:14px;
            line-height:1.52;
        ">

        <div style="
            color:#6f3fa0;
            font-size:17px;
            font-weight:bold;
            margin-bottom:6px;
        ">
        Interpretation
        </div>

        <div style="margin-bottom:6px;">
        For <b>cd(u,k)</b>, the zeros occur at odd multiples
        of K and even multiples of iK′.
        </div>

        <div style="margin-bottom:6px;">
        Its poles occur at odd multiples of both K and iK′.
        </div>

        <div>
        This regular pole-zero lattice is particularly important
        in the construction and interpretation of elliptic filters.
        </div>

        </div>
        """

    # ========================================================
    # TITLE
    # ========================================================

    if mode == 'sn':

        ax.set_title(
            'Zeros and Poles of sn(u,k)',
            fontsize=14,
            fontweight='bold',
            color='#6f3fa0'
        )

    else:

        ax.set_title(
            'Zeros and Poles of cd(u,k)',
            fontsize=14,
            fontweight='bold',
            color='#6f3fa0'
        )

    # ========================================================
    # REDRAW ONLY
    # ========================================================

    fig.canvas.draw_idle()

# ============================================================
# CONNECT CONTROLS
# ============================================================

function_selector.observe(
    update_notebook,
    names='value'
)

k_slider.observe(
    update_notebook,
    names='value'
)

# ============================================================
# INITIAL UPDATE
# ============================================================

update_notebook()

# ============================================================
# COMPLETE NOTEBOOK
# ============================================================

main_layout = VBox(
    [
        documentation,
        top_row,
        visual_row
    ],
    layout=Layout(
        width='1180px',
        gap='10px',
        overflow='visible'
    )
)

display(
    main_layout
)